In [ ]:
!pip install pymupdf
!pip install cohere

!pip install -q langchain-google-genai langchain-qdrant qdrant-client


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 64.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 136.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pymupdf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 13.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cohere]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [ ]:
import os,base64
import pymupdf


In [ ]:
from langchain_core import embeddings
import cohere

### convert PDF to Images
as we are working with the images so convert everyhing into Images

In [ ]:
import pymupdf
from google.colab import userdata
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
    )

In [ ]:
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
    output_dimensionality=1536
)

In [ ]:
import pymupdf

def extract_pdf_text(file_path):
    doc = pymupdf.open(file_path)

    page_texts = []
    page_numbers = []

    # Extract text from every PDF page
    for i, page in enumerate(doc):
        text = page.get_text("text").strip()

        # Skip empty pages
        if text:
            page_texts.append(text)
            page_numbers.append(i + 1)

    doc.close()

    if not page_texts:
        raise ValueError(
            "No text found in the PDF. It may need OCR."
        )

    return page_texts, page_numbers

In [ ]:
def generate_embeddings(page_texts):

    # Convert page text into vectors
    vectors = embeddings.embed_documents(page_texts)

    return vectors

In [ ]:
def pdf_to_embeddings(file_path):

    # Step 1: Extract text and page numbers
    page_texts, page_numbers = extract_pdf_text(file_path)

    # Step 2: Generate embeddings
    vectors = generate_embeddings(page_texts)

    # Step 3: Return all results
    return {
        "texts": page_texts,
        "vectors": vectors,
        "pages": page_numbers
    }


# Execute the function
result = pdf_to_embeddings(
    "/content/Imran.pdf"
)

# Display results
print("Pages processed:", len(result["texts"]))
print("Number of vectors:", len(result["vectors"]))
print("Vector dimensions:", len(result["vectors"][0]))
print("First vector, first 10 values:", result["vectors"][0][:10])

Pages processed: 1
Number of vectors: 1
Vector dimensions: 1536
First vector, first 10 values: [-0.006646058, 0.037128963, 0.011613654, -0.07146296, -0.023533808, 0.004209545, -0.004836912, 0.014960992, -0.018922662, -0.020445874]


In [ ]:
!pip install -qU langchain-qdrant qdrant-client

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

In [ ]:

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# Get the Qdrant database URL and API key from Colab Secrets
QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

# Name of the collection (similar to a table in a traditional database)
COLLECTION_NAME = "this_is_testing_work"

# Each embedding vector must contain 1536 numbers
# This value must match the embedding model you use
DIM = 1536

# Connect to your Qdrant cloud database
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    cloud_inference=True
)

# Check whether the collection already exists
if not client.collection_exists(COLLECTION_NAME):

    # Create a collection to store vectors and their associated data
    client.create_collection(
        collection_name=COLLECTION_NAME,

        # Define the settings for the vectors stored in this collection
        vectors_config=VectorParams(
            size=DIM,  # Number of values in each vector

            # COSINE compares the direction of two vectors
            # It helps find vectors that are semantically similar
            distance=Distance.COSINE
        )
    )

    print("Collection created successfully!")

else:
    print("Collection already exists!")


Collection already exists!


# store data here

In [ ]:

from qdrant_client.models import PointStruct

points = []

# Loop through the actual Gemini embedding vectors
for i, vector in enumerate(result["vectors"]):

    point = PointStruct(
        id=i,  # Unique ID for each vector

        vector=vector,  # Gemini embedding vector

        # Store metadata alongside the vector
        payload={
            "page": result["pages"][i],  # Original PDF page number
            "text": result["texts"][i]   # Text extracted from that page
        }
    )

    # Add this point to our list
    points.append(point)

# Upload all points to Qdrant
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print(f"Successfully stored {len(points)} points!")


Successfully stored 1 points!


In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

In [ ]:

# Connect LangChain to the existing collection
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings
)

# Ask a question about your PDF
question = "imran?"

# Retrieve the three most relevant page records
documents = vector_store.similarity_search(
    question,
    k=3
)

for doc in documents:
    print("Page:", doc.metadata.get("page"))
    print("Text:", doc.page_content[:500])
    print("-" * 40)


Page: None
Text: 
----------------------------------------
Page: None
Text: 
----------------------------------------
Page: None
Text: 
----------------------------------------


In [ ]:

from langchain_core.prompts import ChatPromptTemplate

question = "What is the main topic of this document?"

# Retrieve relevant text from Qdrant
documents = vector_store.similarity_search(
    question,
    k=3
)

# Combine the retrieved text and page numbers
context = "\n\n".join(
    f"Page {doc.metadata.get('page')}:\n{doc.page_content}"
    for doc in documents
)

# Tell Gemini to answer using the retrieved context
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Answer the user's question using only the supplied
        document context. If the context does not contain
        the answer, say that the document does not provide
        enough information. Cite the page numbers when possible."""
    ),
    (
        "human",
        "Document context:\n{context}\n\nQuestion: {question}"
    )
])

chain = prompt | llm

response = chain.invoke({
    "context": context,
    "question": question
})

print(response.content)


[{'type': 'text', 'text': 'The document does not provide enough information to determine its main topic.', 'extras': {'signature': 'EnMKcQFpFH0TSHDbZVrHclZwkbbzKB7QnbpkxTrZKZPge7wQmSaXrkX+gCYKV09BZU6pTvnqKE+eAqZyYNjLisNLzH1N6PKMKNI1WeU+7QoC96aZB42//FJ+y05OUchjww2pfhSm9F1NMmkDb9YYFDhLs4Ni'}}]


# Hybrid Search

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Document

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
cloud_inference=True
COLLECTION_NAME1="hybrid_qdrat"
distance=Distance.COSINE
Dim= 1536
chuk_size=800
checuk_overlap=100
Embed_Model_Name="gemini-embedding-001"

In [ ]:
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
embeddings = GoogleGenerativeAIEmbeddings(
    model=Embed_Model_Name,
    google_api_key=GOOGLE_API_KEY,
    output_dimensionality=Dim
)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Document ,SparseVectorParams,Modifier


In [ ]:
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    cloud_inference=True
)

if not client.collection_exists(COLLECTION_NAME1):
    client.create_collection(
        collection_name=COLLECTION_NAME1,
        vectors_config={
            "dense":VectorParams(
            size=DIM,
            distance=Distance.COSINE
        )},
        sparse_vectors_config={
           "bm25":
           SparseVectorParams(modifier=Modifier.IDF)
        }
    )
    print("Collection created successfully!")
else:
    print("Collection already exists!")


Collection already exists!


In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)

# Combine all extracted page texts into a single string to split, or split page by page
full_text = "\n".join(result["texts"])

texts = text_splitter.split_text(full_text)
print(f"Created {len(texts)} chunks.")
print("First chunk preview:", texts[0] if texts else "None")

Created 48 chunks.
First chunk preview: Imran Ali
Lahore, Pakistan | +92 305 6151224 | imranali.13imran@gmail.com
PROFESSIONAL SUMMARY


In [ ]:
vector = embeddings.embed_documents(texts)

In [ ]:
len(vector[0])

1536

In [ ]:
points = []
for i, vec in enumerate(vector):
    point = PointStruct(
        id=i,
        vector={
            "dense": vec,
            "bm25": Document(text=texts[i], model="Qdrant/bm25"),
        },
        payload={
            "text": texts[i]
        }
    )
    points.append(point)

# COLLECTION_NAME1 is a tuple, so we access index 0
print(f"Prepared {len(points)} points for collection '{COLLECTION_NAME1[0]}'")

Prepared 48 points for collection 'h'


In [ ]:
client.upsert(
  collection_name=COLLECTION_NAME1,
  points=points,
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:

from qdrant_client import QdrantClient, models

def setup_and_upload_hybrid_collection(
    client,
    collection_name,
    texts,
    vectors
):
    """
    Creates a hybrid Qdrant collection
    (Dense vectors + Sparse BM25) and uploads data.
    """

    # 1. Handle collection name
    actual_collection_name = (
        collection_name[0]
        if isinstance(collection_name, tuple)
        else collection_name
    )

    # 2. Create collection if it doesn't exist
    if not client.collection_exists(actual_collection_name):

        client.create_collection(
            collection_name=actual_collection_name,

            vectors_config={
                "dense": models.VectorParams(
                    size=1536,
                    distance=models.Distance.COSINE
                )
            },

            sparse_vectors_config={
                "bm25": models.SparseVectorParams(
                    modifier=models.Modifier.IDF
                )
            }
        )

        print("Collection created successfully!")

    else:
        print("Collection already exists!")

    # 3. Prepare points
    points = []

    for i, vec in enumerate(vectors):

        point = models.PointStruct(
            id=i,

            vector={
                "dense": vec,

                "bm25": models.Document(
                    text=texts[i],
                    model="Qdrant/bm25"
                )
            },

            payload={
                "text": texts[i]
            }
        )

        points.append(point)

    # 4. Upload points to Qdrant
    client.upsert(
        collection_name=actual_collection_name,
        points=points,
        wait=True
    )

    print(
        f"Successfully uploaded {len(points)} points "
        f"to collection '{actual_collection_name}'!"
    )


# 5. Run the function
setup_and_upload_hybrid_collection(
    client=client,
    collection_name=COLLECTION_NAME1,
    texts=texts,
    vectors=vector
)


Collection already exists!
Successfully uploaded 48 points to collection 'hybrid_qdrat'!


In [ ]:

def hybrid_search(client, collection_name, query_text, limit=10):

    results = client.query_points(
        collection_name=collection_name,

        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query_text,
                    model="Qdrant/bm25"
                ),
                using="bm25",
                limit=20
            ),

            models.Prefetch(
                query=embeddings.embed_query(query_text),
                using="dense",
                limit=20
            )
        ],

        query=models.FusionQuery(
            fusion=models.Fusion.RRF
        ),

        limit=limit,
        with_payload=True
    )

    return results.points


# Example
query = "who is imrran ali"

search_results = hybrid_search(
    client,
    COLLECTION_NAME1,
    query,
    limit=10
)

for result in search_results:
    print(result.payload["text"])
    print("Score:", result.score)
    print("-" * 50)


Imran Ali
Lahore, Pakistan | +92 305 6151224 | imranali.13imran@gmail.com
PROFESSIONAL SUMMARY
Score: 1.0
--------------------------------------------------
EDUCATION
Virtual University of Pakistan
2022 – 2026
Score: 0.33333334
--------------------------------------------------
BS in Computer Science (Focus on User Experience)
Lahore, Pakistan
BISE Lahore
2021
Score: 0.25
--------------------------------------------------
ICS (Intermediate in Computer Science)
Lahore, Pakistan
Score: 0.2
--------------------------------------------------
data analysis to AWS deployment.
TECHNICAL SKILLS
Score: 0.16666667
--------------------------------------------------
Forward-thinking AI Engineer specialized in Generative AI, Agentic Systems, and end-to-end LLMOps.
Score: 0.14285715
--------------------------------------------------
• Designed, trained, and validated Machine Learning and Deep Learning models for classification and
Score: 0.125
--------------------------------------------------
• Con

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

In [ ]:

def rag_pipeline(question):

    # Step 1: Retrieve relevant documents
    search_results = hybrid_search(
        client=client,
        collection_name=COLLECTION_NAME1,
        query_text=question,
        limit=5
    )

    # Step 2: Extract retrieved text
    context = "\n\n".join(
        [result.payload["text"] for result in search_results]
    )

    # Step 3: Create prompt
    prompt = f"""
    You are a helpful AI assistant.
    Answer the question using the provided context.

    If the answer is not available in the context,
    clearly say that you don't know.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    # Step 4: Invoke Gemini LLM
    response = llm.invoke(prompt)

    # Step 5: Return answer
    return response.content


In [ ]:
question = "What is the main topic of this document?"

answer = rag_pipeline(question)

print("Answer:")
print(answer)

Answer:
[{'type': 'text', 'text': 'The main topic of this document is AI development, specifically focusing on Generative and Agentic AI, including prompt engineering, RAG systems, agentic frameworks (LangGraph), model metrics, and AI guardrails and security.', 'extras': {'signature': 'EnMKcQFpFH0TR80wxUuCLwgA/WNUV6Wkxy8yvS4Y5yWxWXPOYrS4Y8g2vVBKgrO1kVUgmCHcPG/FRmgz6k572PKfdqmytttPa8iG7T8KuWNRy6aWsHpfgQgpnDVUkb8Hnnhrbd2Xm9CPBQv3OFUU7H2UmVmU'}}]
